# Does industry composition change the wage-growth story?

Average hourly earnings can change because earnings change within industries
or because industries account for different shares of total paid hours.

This analysis asks whether changes in industry composition materially alter
the interpretation of recent private-sector earnings growth.

The workflow uses a dated BLS data snapshot, validates the analytical panel,
reconciles the reconstructed aggregate with published earnings, and produces
reproducible tables and charts.

The results describe broad industry averages—not wage changes for the same
individual workers.

In [1]:
### Project modules and imports

from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, Markdown
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "wage_analysis.py").is_file():
    raise FileNotFoundError(
        "Run Jupyter from the repository root containing wage_analysis.py."
    )

sys.path.insert(0, str(PROJECT_ROOT))

from wage_analysis import (
    SERIES_SPEC,
    load_snapshot,
    audit_coverage,
    candidate_common_window,
    validate_selected_window,
    build_analysis_panel,
    benchmark_for_reconciliation,
    calculate_paid_hours_weights,
    reconcile_headline_earnings,
)


import wage_decomposition as wd
import wage_interpretation as wi
import wage_presentation as wp


### Stage 1 - Data Collection

For this analysis, we collect monthly, seasonally adjusted BLS data on (a) total employment (b) average weekly hours (awh) and (c) average hourly earnings (ahe) across the main private sector employment categories.

A few notes on the implementation:

- The data is originally collected via standard API requests
- Metadata about the specific data series pulled are generated and stored in the SERIES_SPEC dictionary (see below for an example record)
- Once the data has been captured, it is cached and saved, which allows me to rerun the analysis on the same dataset in the future for reproducability

In [2]:
SERIES_SPEC[5]

{'series_id': 'CES2000000003',
 'industry_code': '20000000',
 'industry_name': 'Construction',
 'metric': 'average_hourly_earnings',
 'unit': 'nominal_usd_per_hour',
 'frequency': 'monthly',
 'seasonal_adjustment': 'seasonally_adjusted',
 'employee_population': 'all_employees',
 'role': 'component',
 'source_url': 'https://www.bls.gov/web/empsit/cesseriespub.htm'}

### Not sure about this step

In [3]:
SNAPSHOT_PATH = (
    PROJECT_ROOT
    / "data"
    / "snapshots"
    / "snapshot_20261017T201430156177Z.json"
)

ANALYSIS_START = pd.Timestamp("2019-01-01")
ANALYSIS_END = pd.Timestamp("2026-09-01")

COMPARISON_BASE = pd.Timestamp("2025-09-01")
COMPARISON_END = ANALYSIS_END

FIXED_REFERENCE_DATE = COMPARISON_BASE

### Stage 2 - Data validation

Once the data has been collected, several checks and validation steps are run to ensure data quality.
- The following cell runs creates an audit report for each data series (e.g. are the data types correct, are all months included etc.)
- The audited data is validated (e.g. proper date range, missing categories, invalid values etc), if the data has been corrupted, this step will fail

In [4]:
## Example of a clean dataset

observations, metadata = load_snapshot(SNAPSHOT_PATH)
observations["date"] = pd.to_datetime(observations["date"])

coverage = audit_coverage(
    observations,
    SERIES_SPEC,
    start_date=observations["date"].min(),
    end_date=observations["date"].max(),
)

validate_selected_window(
    coverage,
    start_date=ANALYSIS_START,
    end_date=ANALYSIS_END,
)


True

In [5]:
## Example of a corrupted dataset

observations_invalid, metadata_invalid = load_snapshot(SNAPSHOT_PATH)
observations_invalid["date"] = pd.to_datetime(observations_invalid["date"])

# remove required data
observations_invalid = observations_invalid[observations_invalid!='CES0500000001']

coverage_invalid = audit_coverage(
    observations_invalid,
    SERIES_SPEC,
    start_date=observations["date"].min(),
    end_date=observations["date"].max(),
)

validate_selected_window(
    coverage_invalid,
    start_date=ANALYSIS_START,
    end_date=ANALYSIS_END,
)

False

In [6]:
# Example of a valid data set coverage log

coverage.iloc[0]

series_id                    CES1000000001
industry_code                     10000000
industry_name           Mining and logging
metric                          employment
unit                    thousand_employees
role                             component
first_date             2019-01-01 00:00:00
last_date              2026-09-01 00:00:00
observed_start_date             2019-01-01
observed_end_date               2026-09-01
expected_start_date             2019-01-01
expected_end_date               2026-09-01
observed_count                          93
expected_count                          93
missing_months                          []
status                            complete
Name: 0, dtype: object

### Stage 3 - Prepare data for analysis

Data collected and validated from the BLS remains in a raw state not particularly well suited for analysis.  Specifically, the raw data set only includes an coded series_id, date and data value.

In this step the data is organized into a an analysis friendly set of panel data with employment, average_weekly_hours and average_hourly earnings all stored in the same row.

Also, our analysis explores wage growth at the category level.  We perform a reconcilation step using the headline data.  This allows us to identity any discrepency in the data constrcuted via aggregating the sector wide analysis relative to the BLS report private sector number.

In [7]:
## raw data is not well suited to analysis

observations.iloc[:3]

,series_id,date,value,footnotes
0,CES0500000001,2019-01-01,127552.0,[]
1,CES0500000001,2019-02-01,127542.0,[]
2,CES0500000001,2019-03-01,127726.0,[]


In [8]:
component_panel = build_analysis_panel(
    observations,
    SERIES_SPEC,
    start_date=ANALYSIS_START,
    end_date=ANALYSIS_END,
)

In [9]:
## component_panel is organizes the data in a clear useable format

component_panel.iloc[:3]

,date,industry_code,industry_name,role,employment,average_weekly_hours,average_hourly_earnings,employment_unit,average_weekly_hours_unit,average_hourly_earnings_unit
0,2019-01-01,10000000,Mining and logging,component,745.0,46.4,32.73,thousand_employees,hours_per_week,nominal_usd_per_hour
1,2019-01-01,20000000,Construction,component,7425.0,39.7,30.34,thousand_employees,hours_per_week,nominal_usd_per_hour
2,2019-01-01,30000000,Manufacturing,component,12790.0,40.8,27.25,thousand_employees,hours_per_week,nominal_usd_per_hour


## 

In [10]:
# collect the aggregated BLS numbers for private sector employment
benchmark_for_reconciliation_df = benchmark_for_reconciliation(observations, ANALYSIS_START, ANALYSIS_END)

# compute individual sector weights
weights = calculate_paid_hours_weights(component_panel)

#weight_check = (
#    weights.groupby("date", as_index=False)["weight"]
#    .sum()
#    .rename(columns={"weight": "weight_sum"})
#)

# calculate the discrepency between the reported aggregate,
# and the line-level aggregation we developed for the analysis
reconciliation = reconcile_headline_earnings(
    component_panel,
    weights,
    benchmark_for_reconciliation_df,
)


In [11]:
benchmark_for_reconciliation_df.head()

,date,published_total_private_ahe,published_total_private_employment
0,2019-01-01,27.59,127552.0
1,2019-02-01,27.69,127542.0
2,2019-03-01,27.77,127726.0
3,2019-04-01,27.79,127972.0
4,2019-05-01,27.88,128056.0


In [12]:
reconciliation.head()

,date,published_total_private_ahe,reconstructed_ahe,earnings_residual,published_total_private_employment,component_employment,employment_residual
0,2019-01-01,27.59,27.593163,-0.003163,127552.0,127552.0,0.0
1,2019-02-01,27.69,27.690721,-0.000721,127542.0,127542.0,0.0
2,2019-03-01,27.77,27.766112,0.003888,127726.0,127726.0,0.0
3,2019-04-01,27.79,27.793035,-0.003035,127972.0,127972.0,0.0
4,2019-05-01,27.88,27.877360,0.002640,128056.0,128056.0,0.0


In [13]:
## Weighted earnings aggregate very closely to the BLS aggregation

reconciliation['earnings_residual'].describe()

count    93.000000
mean     -0.000224
std       0.003111
min      -0.006248
25%      -0.002683
50%      -0.000123
75%       0.002212
max       0.006205
Name: earnings_residual, dtype: float64

### Stage 4 - Decompose earnings contributions

Next, we decompose the constributions of each sector to overall changes in wages.

At the beginning of the period, each sector makes up a percentage of employment.  The change in overall private sector earning from one period to the next can be decomposed into wage changes within a sector or via employment shifting from sectors with different earnings levels.

In [14]:
# Adapt the Stage 2 column name for the decomposition functions.
stage3_panel = component_panel.rename(
    columns={"average_hourly_earnings": "ahe"}
)

# 1. Monthly and annual decompositions.
monthly_contributions, monthly_summary = wd.build_decomposition_series(
    stage3_panel,
    weights,
    benchmark_for_reconciliation_df,
    lag=1,
)

annual_contributions, annual_summary = wd.build_decomposition_series(
    stage3_panel,
    weights,
    benchmark_for_reconciliation_df,
    lag=12,
)

# 2. Earnings series with paid-hours weights held at the reference month.
fixed_composition = wd.build_fixed_composition_series(
    stage3_panel,
    weights,
    benchmark_for_reconciliation_df,
    reference_date=FIXED_REFERENCE_DATE,
)

KeyError: "['industry_name'] not in index"

### Result Explaination

The data below shows the change in average hourly earnings (ahe) between August 2026 and September 2026.  Industry code 

In [15]:
monthly_contributions.iloc[-10:]

NameError: name 'monthly_contributions' is not defined

In [ ]:
# 3. Attach industry names before building interpretation tables and charts.
if "industry_name" not in annual_contributions.columns:
    industry_names = component_panel[
        ["industry_code", "industry_name"]
    ].drop_duplicates()

    annual_contributions = annual_contributions.merge(
        industry_names,
        on="industry_code",
        how="left",
        validate="many_to_one",
    )

# 4. Summarize the selected annual comparison.
interpretation = wi.build_interpretation_summary(
    annual_summary,
    fixed_composition,
    end_date=COMPARISON_END,
)

materiality = wi.summarize_composition_materiality(
    interpretation,
    annual_contributions,
    end_date=COMPARISON_END,
)

contributors = wi.build_composition_contributor_tables(
    annual_contributions,
    base_date=COMPARISON_BASE,
    end_date=COMPARISON_END,
    top_n=5,
)

# 5. Prepare chart data from the same analytical results.
headline_data = wp.build_headline_comparison_chart_data(
    annual_summary,
    fixed_composition,
    base_date=COMPARISON_BASE,
    end_date=COMPARISON_END,
)

contribution_data = wp.build_composition_contribution_chart_data(
    annual_contributions,
    base_date=COMPARISON_BASE,
    end_date=COMPARISON_END,
)

In [ ]:
s = interpretation.iloc[0]

display(Markdown(f"""
## Main result

**Comparison:** {s["base_date"]:%B %Y} to {s["end_date"]:%B %Y}

Reconstructed nominal average hourly earnings changed by
**{s["reconstructed_change"]:+.3f} dollars per hour**, comprising:

- **Within-industry earnings:** {s["within_total"]:+.3f} dollars per hour.
- **Industry composition:** {s["composition_total"]:+.3f} dollars per hour.

The published total-private measure changed by
**{s["published_change"]:+.3f} dollars per hour**.

Fixed-composition weights are taken from
**{s["reference_date"]:%B %Y}**.
"""))

Take Away: Industry composition modestly restrained aggregate earnings growth over this interval.  Most of the increase reflected higher average earnings within broad industries

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5), constrained_layout=True)

wp.plot_headline_comparison_chart(headline_data, ax=ax)

ax.set_title(
    "Change in Average Hourly Earnings\n"
    f"{COMPARISON_BASE:%b %Y} to {COMPARISON_END:%b %Y}"
)
ax.set_ylabel("Change in nominal AHE ($/hour)")
ax.set_xlabel(f"Fixed-composition weights: {COMPARISON_BASE:%b %Y}")

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6), constrained_layout=True)

wp.plot_composition_contribution_chart(contribution_data, ax=ax)

net_effect = contribution_data["composition_contribution"].sum()

ax.set_title(
    "Industry Composition Contributions\n"
    f"{COMPARISON_BASE:%b %Y} to {COMPARISON_END:%b %Y}"
    f" | Net effect: {net_effect:+.3f} $/hour"
)
ax.set_xlabel("Contribution to nominal AHE change ($/hour)")

plt.show()

In [ ]:
# Cell 12 — Run summary and validation

# Confirm coverage for the explicitly selected analysis window.
validate_selected_window(
    coverage,
    start_date=ANALYSIS_START,
    end_date=ANALYSIS_END,
)

run_summary = pd.Series(
    {
        "Snapshot": SNAPSHOT_PATH.name,
        "Retrieved (UTC)": metadata.get("retrieval_end_utc", "Not recorded"),
        "Analysis window": (
            f"{ANALYSIS_START:%b %Y} to {ANALYSIS_END:%b %Y}"
        ),
        "Headline comparison": (
            f"{COMPARISON_BASE:%b %Y} to {COMPARISON_END:%b %Y}"
        ),
        "Fixed-weight reference": f"{FIXED_REFERENCE_DATE:%b %Y}",
        "Selected-window coverage": "Validated",
        "Required series": len(coverage),
        "Component industries": component_panel["industry_code"].nunique(),
        "Months": component_panel["date"].nunique(),
        "Industry-month rows": len(component_panel),
    },
    name="Value",
).to_frame()

display(Markdown("## Data and validation"))
display(run_summary)

# Recompute arithmetic balances from the outputs.
weight_error = (
    weights.groupby("date")["weight"].sum(min_count=1) - 1
)

monthly_error = (
    monthly_summary["reconstructed_change"]
    - monthly_summary["within_total"]
    - monthly_summary["composition_total"]
)

annual_error = (
    annual_summary["reconstructed_change"]
    - annual_summary["within_total"]
    - annual_summary["composition_total"]
)

# These are numerical tolerances, not economic materiality thresholds.
check_inputs = [
    ("Weights sum to one", weight_error, "Share", 1e-10),
    ("Monthly decomposition balances", monthly_error, "USD/hour", 1e-10),
    ("Annual decomposition balances", annual_error, "USD/hour", 1e-10),
]

checks = pd.DataFrame(
    [
        {
            "Check": label,
            "Maximum absolute error": errors.abs().max(skipna=False),
            "Unit": unit,
            "Tolerance": tolerance,
        }
        for label, errors, unit, tolerance in check_inputs
    ]
).set_index("Check")

checks["Status"] = (
    checks["Maximum absolute error"]
    .le(checks["Tolerance"])
    .map({True: "PASS", False: "CHECK"})
)

display(Markdown("### Arithmetic checks across the analysis window"))
display(
    checks.style.format(
        {
            "Maximum absolute error": "{:.2e}",
            "Tolerance": "{:.0e}",
        }
    )
)

# Benchmark differences are separate from decomposition arithmetic.
benchmark_gaps = pd.DataFrame(
    {
        "Maximum absolute gap": reconciliation[
            ["earnings_residual", "employment_residual"]
        ].abs().max(skipna=False),
        "Unit": ["USD/hour", "Thousand employees"],
    }
).rename(
    index={
        "earnings_residual": "Published vs reconstructed AHE",
        "employment_residual": "Published vs component employment",
    }
)

display(Markdown("### Reconciliation with published totals"))
display(
    benchmark_gaps.style.format(
        {"Maximum absolute gap": "{:.6f}"}
    )
)

display(Markdown(
    "Arithmetic errors should be close to zero within numerical precision. "
    "Benchmark gaps measure agreement with the published totals and should "
    "be assessed separately. These checks establish accounting consistency; "
    "they do not establish economic causality or replace the test suite."
))

if checks["Status"].ne("PASS").any():
    raise ValueError("An arithmetic check failed; review the table above.")